# YOLO11-seg 재학습 (v2)

7개 클래스 인스턴스 세그멘테이션.

## v1 대비 바뀐 점

| # | 문제 | 조치 |
|---|------|------|
| 1 | 시험 문제가 답이랑 같이 새어나감 — train/valid/test가 같은 영상의 거의 똑같은 프레임을 섞어써서, 점수(mAP 0.94~1.00)가 "실력"이 아니라 "암기"를 측정하고 있었음 | 영상을 시간 순으로 잘라 train과 확실히 떨어진 구간만 valid/test로 사용 → 이제 나오는 점수가 진짜 실력 |
| 2 | 인형+네일을 같이 본 적이 없음 — 한 사진에 같이 나온 경우가 0장이라 구분하는 법을 못 배움 | `copy_paste` 증강으로 학습 중 다른 사진의 물체를 합성해 이런 조합을 인위적으로 만들어 보여줌 |
| 3 | 작은 물체(nail)가 배경 오탐의 44%를 차지 | scale 증강을 넓혀 같은 작은 크기에서도 더 다양한 배경/각도의 예시를 학습 |
| 4 | 오탐이 잘 나는 클래스에 맞는 기준이 없었음 | 클래스별 최적 confidence 임계값을 자동으로 찾아 `class_thresholds.json`으로 저장 |

## 실행 방법

1. `perception_train` 폴더(이 노트북 + `merged_dataset.zip`)를 **내 드라이브 최상위**에 업로드
2. 런타임 > 런타임 유형 변경 > **T4 GPU**
3. 런타임 > **모두 실행**

결과 이미지는 드라이브의 `perception_train/result/` 에 저장됩니다.

## 1. 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설정

여기만 고치면 됩니다.

In [ ]:
from pathlib import Path

# ── 드라이브에 업로드한 프로젝트 폴더 ─────────────────────────────────
PROJECT_DIR = Path('/content/drive/MyDrive/perception_train')

# ── 모델 / 학습 ──────────────────────────────────────────────────────
MODEL    = 'yolo11n-seg.pt'   # 정확도 우선이면 'yolo11s-seg.pt' (권장, 아래 주석 참고)
EPOCHS   = 150
IMGSZ    = 832                # 원본이 640이라 832 이상은 효용 체감. OOM/시간 부족하면 640
BATCH    = 8                  # IMGSZ=832 기준 T4 안전값. 640이면 16 가능
PATIENCE = 30
RUN_NAME = 'exp_v2'

# ── 시간축 재분할 (데이터 누수 제거) ──────────────────────────────────
BLOCK_SIZE = 20   # 연속 프레임을 이 크기 블록으로 잘라 블록 단위로 split 배정
GAP        = 3    # valid/test 블록 양끝에서 버릴 프레임 수 (train과의 시간 완충)
CYCLE      = 6    # block_index % CYCLE 로 split 배정
VAL_MOD    = 4    # -> valid
TEST_MOD   = 5    # -> test  (나머지는 train. 실측 train 1680 / valid 230 / test 193)

SEED = 0

# ── 파생 경로 ────────────────────────────────────────────────────────
ZIP_PATH   = str(PROJECT_DIR / 'merged_dataset.zip')
RESULT_DIR = PROJECT_DIR / 'result'
RUNS_DIR   = PROJECT_DIR / 'runs'
SRC_DIR    = Path('/content/data/merged_dataset')   # 압축 해제 위치
DST_DIR    = Path('/content/dataset_ts')            # 재분할된 학습 데이터셋
DATA_YAML  = DST_DIR / 'data.yaml'

assert PROJECT_DIR.exists(), f'프로젝트 폴더를 찾을 수 없습니다: {PROJECT_DIR}'
assert Path(ZIP_PATH).exists(), f'merged_dataset.zip 을 찾을 수 없습니다: {ZIP_PATH}'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('프로젝트 :', PROJECT_DIR)
print('결과 저장:', RESULT_DIR)

## 3. 패키지 설치 + 데이터셋 압축 해제

In [ ]:
!pip install -q ultralytics

In [ ]:
!rm -rf /content/data /content/dataset_ts
!unzip -q -o "$ZIP_PATH" -d /content/data
!find /content/data -name "*.cache" -delete

import glob
print('이미지:', len(glob.glob('/content/data/**/*.jpg', recursive=True)))
print('라벨  :', len(glob.glob('/content/data/**/*.txt', recursive=True)))

## 4. 시간축 재분할 — 데이터 누수 제거 ⭐

**가장 중요한 단계입니다.**

기존 split은 Roboflow가 프레임을 랜덤 배정해서, 사실상 같은 장면이 train과 test에 동시에
들어가 있었습니다. 여기서는 영상별로 프레임을 시간 순 정렬한 뒤 `BLOCK_SIZE` 단위 연속
블록으로 잘라 **블록 통째로** split에 배정하고, valid/test 블록의 양 끝 `GAP` 프레임을
버려 train 프레임과 시간적으로 떨어뜨립니다.

블록을 흩어서 배정하기 때문에 valid/test가 영상의 특정 구간에만 몰리지 않습니다.

In [ ]:
import re, shutil, json
from collections import defaultdict
import yaml

# 원본 클래스 이름
with open(SRC_DIR / 'data.yaml') as f:
    src_yaml = yaml.safe_load(f)
CLASS_NAMES = list(src_yaml['names'])
NC = len(CLASS_NAMES)
print('클래스:', NC, CLASS_NAMES)

# 파일명 예: ringring_gel_nail_..._video_nail_mp4-0248_jpg.rf.<hash>.jpg
#            -> 영상 = "..._video_nail_mp4", 프레임 = 248
PAT = re.compile(r'^(.*?_mp4)-(\d+)_jpg')

by_video = defaultdict(list)
unmatched = []
for sp in ['train', 'valid', 'test']:
    for img in sorted((SRC_DIR / sp / 'images').glob('*')):
        m = PAT.match(img.name)
        if not m:
            unmatched.append(img.name)
            continue
        lbl = SRC_DIR / sp / 'labels' / (img.stem + '.txt')
        by_video[m.group(1)].append((int(m.group(2)), img, lbl))

if unmatched:
    print(f'경고: 파일명 패턴 불일치 {len(unmatched)}건 (train으로 편입)', unmatched[:3])

# 블록 단위 배정
assign = {'train': [], 'valid': [], 'test': []}
for video, items in sorted(by_video.items()):
    items.sort(key=lambda x: x[0])                    # 프레임 번호 순
    nblocks = (len(items) + BLOCK_SIZE - 1) // BLOCK_SIZE
    for bi in range(nblocks):
        block = items[bi * BLOCK_SIZE:(bi + 1) * BLOCK_SIZE]
        r = bi % CYCLE
        sp = 'valid' if r == VAL_MOD else ('test' if r == TEST_MOD else 'train')
        if sp != 'train':
            # 양 끝 GAP 프레임을 버려 train 블록과 시간적으로 분리
            block = block[GAP:len(block) - GAP] if len(block) > 2 * GAP else []
        for fr, img, lbl in block:
            assign[sp].append((video, fr, img, lbl))

# 파일 복사
for sp in ['train', 'valid', 'test']:
    (DST_DIR / sp / 'images').mkdir(parents=True, exist_ok=True)
    (DST_DIR / sp / 'labels').mkdir(parents=True, exist_ok=True)
    for video, fr, img, lbl in assign[sp]:
        shutil.copy2(img, DST_DIR / sp / 'images' / img.name)
        if lbl.exists():
            shutil.copy2(lbl, DST_DIR / sp / 'labels' / lbl.name)
        else:  # 라벨 없는 이미지 = 배경 네거티브. 빈 txt 생성
            (DST_DIR / sp / 'labels' / (img.stem + '.txt')).touch()

# 선택: PROJECT_DIR/backgrounds/ 에 물체가 전혀 없는 배경 사진을 넣어두면
#       배경 네거티브로 train에 합쳐집니다 (오탐 억제에 매우 효과적).
BG_DIR = PROJECT_DIR / 'backgrounds'
n_bg = 0
if BG_DIR.exists():
    for img in sorted(BG_DIR.glob('*')):
        if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue
        shutil.copy2(img, DST_DIR / 'train' / 'images' / img.name)
        (DST_DIR / 'train' / 'labels' / (img.stem + '.txt')).touch()
        n_bg += 1
print(f'배경 네거티브 이미지: {n_bg}장')

# data.yaml 작성
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(DST_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': NC,
    'names': CLASS_NAMES,
}, sort_keys=False, allow_unicode=True))

for sp in ['train', 'valid', 'test']:
    print(f'{sp:6}: {len(list((DST_DIR / sp / "images").glob("*")))}장')

### 4-1. 누수 제거 검증

In [ ]:
# valid/test 프레임이 train 프레임과 몇 프레임 떨어져 있는지 재확인
frames = {sp: defaultdict(set) for sp in ['train', 'valid', 'test']}
for sp in ['train', 'valid', 'test']:
    for video, fr, img, lbl in assign[sp]:
        frames[sp][video].add(fr)

leak_report = {}
print(f'{"split":6} {"±2 이내 인접":>14} {"최소 시간거리":>14}')
for sp in ['valid', 'test']:
    leaked = total = 0
    min_dist = 10 ** 9
    for video, frs in frames[sp].items():
        tr = frames['train'][video]
        if not tr:
            continue
        for fr in frs:
            total += 1
            d = min(abs(fr - t) for t in tr)
            min_dist = min(min_dist, d)
            if d <= 2:
                leaked += 1
    pct = leaked / total * 100 if total else 0
    leak_report[sp] = {'leaked': leaked, 'total': total, 'pct': pct, 'min_dist': min_dist}
    print(f'{sp:6} {leaked:6d}/{total:<6d} ({pct:4.1f}%) {min_dist:>14d}')

assert all(v['pct'] == 0 for v in leak_report.values()), '누수가 남아 있습니다. GAP을 키우세요.'
print('\n누수 제거 완료 (이전: valid 100%, test 99.1%)')

## 5. 데이터셋 진단 이미지 저장

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 그래프 텍스트는 영문 사용 (Colab에 한글 폰트가 없어 깨짐 방지)
plt.rcParams['figure.dpi'] = 130

def poly_area(vals):
    xs, ys = vals[0::2], vals[1::2]
    n = len(xs)
    a = sum(xs[i] * ys[(i + 1) % n] - xs[(i + 1) % n] * ys[i] for i in range(n))
    return abs(a) / 2

# 통계 수집
inst_counts = {sp: np.zeros(NC, int) for sp in ['train', 'valid', 'test']}
areas_single, areas_mix = defaultdict(list), defaultdict(list)
cooccur = np.zeros((NC, NC), int)

for sp in ['train', 'valid', 'test']:
    for lbl in (DST_DIR / sp / 'labels').glob('*.txt'):
        is_mix = lbl.name.startswith('mix')
        present = set()
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) < 7:
                continue
            c = int(p[0])
            coords = list(map(float, p[1:]))
            inst_counts[sp][c] += 1
            present.add(c)
            (areas_mix if is_mix else areas_single)[c].append(poly_area(coords) * 100)
        if sp == 'train':
            for a in present:
                for b in present:
                    cooccur[a][b] += 1

short = [n[:16] for n in CLASS_NAMES]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

# (1) split별 인스턴스 수
ax = axes[0]
x = np.arange(NC)
bottom = np.zeros(NC)
for sp, col in zip(['train', 'valid', 'test'], ['#2b6cb0', '#63b3ed', '#f6ad55']):
    ax.bar(x, inst_counts[sp], bottom=bottom, label=sp, color=col)
    bottom += inst_counts[sp]
ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_title('Instances per class (after temporal re-split)')
ax.set_ylabel('instances'); ax.legend()

# (2) 스케일 분포: 단일샷 vs mix
ax = axes[1]
w = 0.38
ax.bar(x - w / 2, [np.mean(areas_single[c]) if areas_single[c] else 0 for c in range(NC)],
       w, label='single-object shots', color='#2b6cb0')
ax.bar(x + w / 2, [np.mean(areas_mix[c]) if areas_mix[c] else 0 for c in range(NC)],
       w, label='mix (multi-object) shots', color='#e53e3e')
ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_title('Mean object area (% of image)')
ax.set_ylabel('area %'); ax.legend()

# (3) 클래스 동시 등장 (train)
ax = axes[2]
cm = cooccur.astype(float).copy()
np.fill_diagonal(cm, 0)
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(x); ax.set_xticklabels(short, rotation=90, fontsize=7)
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=7)
ax.set_title('Class co-occurrence in same image (train)')
for i in range(NC):
    for j in range(NC):
        if i != j:
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=7,
                    color='white' if cm[i, j] > cm.max() * 0.5 else '#444')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(RESULT_DIR / '01_dataset_diagnostics.png', bbox_inches='tight')
plt.close()
print('저장:', RESULT_DIR / '01_dataset_diagnostics.png')
print('\ndoll x nail 동시 등장 횟수 (train):', cooccur[3][6],
      '  <- 0이면 copy_paste mixup 증강이 이 공백을 메웁니다')

## 6. 학습

증강 설정 의도:

- **`copy_paste=0.5, copy_paste_mode='mixup'`** — 다른 이미지의 인스턴스를 현재 이미지에
  합성. 학습 데이터에 doll+nail 조합이 0건인 문제를 직접 메웁니다. (`flip` 모드는 같은
  이미지 안에서만 복제하므로 클래스 조합을 못 만듭니다. 반드시 `mixup`)
- **`mosaic=1.0`** — 4장을 이어붙여 새로운 클래스 조합 + 스케일 변화 생성
- **`scale=0.75`** — 배율 0.25~1.75배. nail이 mix 장면에서만 작게(0.65%) 학습된 스케일
  편중을 완화
- **`degrees=15, translate=0.15`** — 영상 11개 = 사실상 11개 장면뿐이므로 기하 증강 강화
- **`close_mosaic=15`** — 마지막 15에포크는 증강을 끄고 실제 분포로 마무리

In [ ]:
from ultralytics import YOLO
import torch

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (T4로 변경하세요)')

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,

    # ── 증강 ──────────────────────────────────────────────
    copy_paste=0.5,
    copy_paste_mode='mixup',   # 핵심: 다른 이미지의 객체를 합성해 없던 클래스 조합 생성
    mosaic=1.0,
    close_mosaic=15,
    scale=0.75,
    degrees=15.0,
    translate=0.15,
    fliplr=0.5,
    flipud=0.0,                # 제품 상하반전은 비현실적이라 끔
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,

    cos_lr=True,
    plots=True,
)

RUN_PATH = Path(results.save_dir)
BEST = RUN_PATH / 'weights' / 'best.pt'
print('\n학습 완료:', RUN_PATH)
print('가중치    :', BEST)

## 7. test 셋 평가 (누수 없는 정직한 지표)

v1의 mAP는 누수 때문에 부풀려진 값이었습니다. 여기서 나오는 숫자가 **실제 기대 성능**에
훨씬 가깝습니다. 숫자가 v1보다 낮게 나오는 것이 정상이며, 오히려 신뢰할 수 있는 값입니다.

In [ ]:
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ,
                   project=str(RUNS_DIR), name=RUN_NAME + '_test', exist_ok=True, plots=True)
VAL_PATH = Path(metrics.save_dir)

print(f"\n{'class':<38} {'box mAP50':>10} {'box mAP50-95':>14} {'mask mAP50':>11}")
print('-' * 76)
for i, c in enumerate(metrics.box.ap_class_index):
    print(f'{CLASS_NAMES[c]:<38} {metrics.box.ap50[i]:>10.3f} {metrics.box.ap[i]:>14.3f} '
          f'{metrics.seg.ap50[i]:>11.3f}')
print('-' * 76)
print(f'{"ALL":<38} {metrics.box.map50:>10.3f} {metrics.box.map:>14.3f} {metrics.seg.map50:>11.3f}')

## 8. 오탐 분석 + 클래스별 최적 conf 임계값 자동 탐색 ⭐

test 셋 전체를 아주 낮은 conf(0.01)로 추론한 뒤, 예측을 GT와 IoU 0.5로 매칭해서
세 종류로 분류합니다.

- **TP** — 같은 클래스의 GT와 매칭
- **class-FP** — 다른 클래스의 GT와 매칭 (클래스 혼동)
- **background-FP** — 어떤 GT와도 매칭 안 됨 (허공에 검출 = v1에서 nail이 44% 차지)

그다음 클래스마다 F1을 최대화하는 conf 임계값을 찾아 `class_thresholds.json` 으로 저장합니다.

In [ ]:
from PIL import Image

def polys_to_boxes(lbl_path, W, H):
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().splitlines():
        p = line.split()
        if len(p) < 7:
            continue
        c = int(p[0]); v = list(map(float, p[1:]))
        xs, ys = v[0::2], v[1::2]
        boxes.append((c, min(xs) * W, min(ys) * H, max(xs) * W, max(ys) * H))
    return boxes

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0., ix2 - ix1), max(0., iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.

test_imgs = sorted((DST_DIR / 'test' / 'images').glob('*'))
preds = []            # (pred_cls, conf, matched_gt_cls)  matched_gt_cls=-1 -> background
gt_total = np.zeros(NC, int)

for img_path in test_imgs:
    W, H = Image.open(img_path).size
    gts = polys_to_boxes(DST_DIR / 'test' / 'labels' / (img_path.stem + '.txt'), W, H)
    for c, *_ in gts:
        gt_total[c] += 1

    r = best.predict(str(img_path), conf=0.01, iou=0.6, imgsz=IMGSZ, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        continue
    order = np.argsort(-r.boxes.conf.cpu().numpy())
    used = set()
    for i in order:
        pb = r.boxes.xyxy[i].cpu().numpy()
        pc = int(r.boxes.cls[i]); conf = float(r.boxes.conf[i])
        bestj, bestiou = -1, 0.5          # IoU 0.5 미만은 미매칭
        for j, g in enumerate(gts):
            if j in used:
                continue
            v = iou(pb, g[1:])
            if v >= bestiou:
                bestj, bestiou = j, v
        if bestj >= 0:
            used.add(bestj)
            preds.append((pc, conf, gts[bestj][0]))
        else:
            preds.append((pc, conf, -1))

preds = np.array(preds, dtype=float).reshape(-1, 3)   # 예측이 0건이어도 shape 유지
assert len(preds) > 0, '예측이 하나도 없습니다. 학습이 실패했는지 확인하세요.'
print(f'test 이미지 {len(test_imgs)}장, GT {gt_total.sum()}개, 예측 후보 {len(preds)}개')

In [ ]:
# 클래스별 F1 최대 임계값 탐색
grid = np.arange(0.05, 0.96, 0.01)
best_thr, f1_curves = {}, {}

for c in range(NC):
    sel = preds[preds[:, 0] == c]
    f1s = []
    for t in grid:
        s = sel[sel[:, 1] >= t]
        tp = int((s[:, 2] == c).sum())
        fp = len(s) - tp
        fn = gt_total[c] - tp
        prec = tp / (tp + fp) if tp + fp else 0.
        rec = tp / (tp + fn) if tp + fn else 0.
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.)
    f1s = np.array(f1s)
    f1_curves[c] = f1s
    best_thr[c] = float(grid[int(f1s.argmax())])

with open(RESULT_DIR / 'class_thresholds.json', 'w') as f:
    json.dump({'class_names': CLASS_NAMES,
               'thresholds': {str(c): best_thr[c] for c in range(NC)}}, f, indent=2)

print(f'{"class":<38} {"best conf":>10} {"best F1":>9}')
print('-' * 60)
for c in range(NC):
    print(f'{CLASS_NAMES[c]:<38} {best_thr[c]:>10.2f} {f1_curves[c].max():>9.3f}')

In [ ]:
# 오탐 구성 + 임계값 곡선 그래프
def breakdown(thr_map):
    tp = np.zeros(NC, int); cfp = np.zeros(NC, int); bfp = np.zeros(NC, int)
    for pc, conf, gc in preds:
        pc = int(pc); gc = int(gc)
        if conf < thr_map[pc]:
            continue
        if gc == pc:   tp[pc] += 1
        elif gc == -1: bfp[pc] += 1
        else:          cfp[pc] += 1
    return tp, cfp, bfp

flat25 = {c: 0.25 for c in range(NC)}
tp_a, cfp_a, bfp_a = breakdown(flat25)
tp_b, cfp_b, bfp_b = breakdown(best_thr)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

ax = axes[0]
for c in range(NC):
    ax.plot(grid, f1_curves[c], label=short[c], lw=1.6)
    ax.axvline(best_thr[c], color='gray', alpha=0.15)
ax.set_xlabel('confidence threshold'); ax.set_ylabel('F1')
ax.set_title('F1 vs conf threshold (per class)')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1]
w = 0.38
ax.bar(x - w / 2, bfp_a, w, label='conf=0.25 (flat)', color='#e53e3e')
ax.bar(x + w / 2, bfp_b, w, label='per-class optimal', color='#2b6cb0')
ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_title(f'Background false positives\n(total {bfp_a.sum()} -> {bfp_b.sum()})')
ax.set_ylabel('count'); ax.legend()

ax = axes[2]
ax.bar(x, tp_b, label='TP', color='#2b6cb0')
ax.bar(x, cfp_b, bottom=tp_b, label='class-confusion FP', color='#f6ad55')
ax.bar(x, bfp_b, bottom=tp_b + cfp_b, label='background FP', color='#e53e3e')
ax.plot(x, gt_total, 'k_', ms=18, label='ground truth count')
ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_title('Detection breakdown at per-class optimal threshold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULT_DIR / '04_fp_analysis.png', bbox_inches='tight')
plt.close()

print(f'배경 오탐: conf=0.25 -> {bfp_a.sum()}건 / 클래스별 최적 임계값 -> {bfp_b.sum()}건')
print(f'클래스 혼동 오탐: {cfp_a.sum()}건 -> {cfp_b.sum()}건')
print('저장:', RESULT_DIR / '04_fp_analysis.png')

## 9. 정성 확인 — 다물체 장면 예측 결과

In [ ]:
import math

# GT 객체가 많은 test 이미지 우선 선택 (다물체 장면)
scored = []
for p in test_imgs:
    n = len([l for l in (DST_DIR / 'test' / 'labels' / (p.stem + '.txt')).read_text().splitlines() if l.strip()])
    scored.append((n, p))
scored.sort(key=lambda t: -t[0])
picks = [p for _, p in scored[:8]]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, p in zip(axes.ravel(), picks):
    r = best.predict(str(p), conf=0.01, iou=0.6, imgsz=IMGSZ, verbose=False)[0]
    keep = [i for i in range(len(r.boxes))
            if float(r.boxes.conf[i]) >= best_thr[int(r.boxes.cls[i])]]
    ax.imshow(r[keep].plot(labels=True, conf=True)[..., ::-1])
    n_gt = len([l for l in (DST_DIR / 'test' / 'labels' / (p.stem + '.txt')).read_text().splitlines() if l.strip()])
    ax.set_title(f'GT={n_gt}  pred={len(keep)}', fontsize=10)
    ax.axis('off')
for ax in axes.ravel()[len(picks):]:
    ax.axis('off')
plt.suptitle('Predictions with per-class optimal thresholds (test set)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULT_DIR / '05_predictions.png', bbox_inches='tight')
plt.close()
print('저장:', RESULT_DIR / '05_predictions.png')

## 10. 결과 정리 → `result/` 폴더

In [ ]:
import datetime

# 학습/검증 산출물 중 중요한 것만 복사
copy_map = [
    (RUN_PATH / 'results.png',                       '02_training_curves.png'),
    (VAL_PATH / 'confusion_matrix_normalized.png',   '03_confusion_matrix.png'),
    (VAL_PATH / 'BoxPR_curve.png',                   '06_box_pr_curve.png'),
    (VAL_PATH / 'MaskPR_curve.png',                  '07_mask_pr_curve.png'),
    (RUN_PATH / 'train_batch0.jpg',                  '08_augmented_batch.jpg'),
]
for src, dst in copy_map:
    if src.exists():
        shutil.copy2(src, RESULT_DIR / dst)
        print('저장:', dst)
    else:
        print('없음:', src.name)

# best.pt 도 결과 폴더에 사본 보관
shutil.copy2(BEST, RESULT_DIR / 'best.pt')

summary = {
    'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'model': MODEL, 'imgsz': IMGSZ, 'epochs_cfg': EPOCHS, 'batch': BATCH,
    'split': {sp: len(assign[sp]) for sp in ['train', 'valid', 'test']},
    'background_negatives': n_bg,
    'leakage_after_resplit': leak_report,
    'test_metrics': {
        'box_mAP50': float(metrics.box.map50), 'box_mAP50_95': float(metrics.box.map),
        'mask_mAP50': float(metrics.seg.map50), 'mask_mAP50_95': float(metrics.seg.map),
    },
    'per_class_threshold': {CLASS_NAMES[c]: best_thr[c] for c in range(NC)},
    'background_fp': {'flat_conf_0.25': int(bfp_a.sum()), 'per_class_optimal': int(bfp_b.sum())},
}
(RESULT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))

print('\n=== 요약 ===')
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\n모든 결과:', RESULT_DIR)

## 11. 추론 시 사용법

`result/class_thresholds.json` 의 클래스별 임계값을 적용해야 오탐이 억제됩니다.

```python
import json
from ultralytics import YOLO

model = YOLO('best.pt')
THR = {int(k): v for k, v in json.load(open('class_thresholds.json'))['thresholds'].items()}

r = model.predict(source, conf=0.01, iou=0.6, imgsz=832, verbose=False)[0]
keep = [i for i in range(len(r.boxes))
        if float(r.boxes.conf[i]) >= THR[int(r.boxes.cls[i])]]
r = r[keep]
```

## 12. 다음 단계 (효과 큰 순서)

1. **배경 네거티브 촬영** — 대상 제품이 하나도 없는 상태의 촬영 환경 사진 100~200장을
   드라이브의 `perception_train/backgrounds/` 에 넣고 재실행하면 자동으로 학습에 포함됩니다.
   배경 오탐(현재 최대 문제)에 가장 직접적인 처방입니다.
2. **새 영상 추가 촬영** — 지금 데이터는 영상 11개 = 사실상 장면 11개뿐입니다. 프레임 수를
   늘려도 다양성은 늘지 않습니다. 특히 **doll과 nail을 함께 놓은 영상**, 그리고 조명/배경/
   각도를 바꾼 영상이 필요합니다.
3. **Roboflow에서 더 높은 해상도로 재export** — 현재 데이터셋은 640×640으로 다운스케일되어
   nail이 33px밖에 안 됩니다. 원본 영상이 1080p라면 1280으로 재export하는 것만으로 작은
   물체 성능이 크게 개선됩니다.
4. **`MODEL = 'yolo11s-seg.pt'`** — 파라미터 2.9M → 10M. 클래스 판별력이 눈에 띄게 좋아집니다.
   엣지 디바이스 제약이 없다면 권장.